# ODEs: Damped driven pendulum

We are now going to investigate a damped driven pendulum. The equations of motion for a pendulum with length $L$ and mass $m$, with damping coefficient $b$, undergoing forced motion with magnitude $F_D$ and driving frequency $\omega_D$ are

$$
\frac{d^2\theta}{dt^2} = -\omega_0^2 \sin(\theta) - \Gamma \frac{d\theta}{dt} + F_D \cos(\omega_D t)
$$

where $\omega_0 = \sqrt{g/L}$ (with $g$ being the gravitational constant) and $\Gamma = b/2m$. 

For small oscillations, the analytic steady-state solution is

$$
\theta(t) = \frac{F_D \cos(\omega_D t + \phi)}{\sqrt{(\omega_0^2 - \omega_D^2)^2 + (\Gamma \omega_D)^2}}
$$

However, this fails for large angles because it is a nonlinear oscillator. For that we will need numeric solutions. 

For all cases, it is insufficient to utilize the [Euler](https://en.wikipedia.org/wiki/Euler_method), [Euler-Cromer](https://en.wikipedia.org/wiki/Semi-implicit_Euler_method), or [Midpoint](https://en.wikipedia.org/wiki/Midpoint_method) methods to achieve stable orbits. We could use the [Verlet](https://en.wikipedia.org/wiki/Verlet_integration) scheme:

$$
\vec{v}_n = \frac{\vec{r}_{n+1} - \vec{r}_{n-1}}{2\tau} + \mathcal{O}(\tau^2)
$$

$$
\vec{x}_n = 2\vec{x}_{n+1} - \vec{x}_{n-1} + \tau^2 \vec{a}_n
$$


But instead, from here, let's just use `scipy`'s ODE integration tools. The "native python" implementation from previous notebooks is very slow, and we need a lot of computations. We could write a C++ implementation and put it into python via pybind etc, but in this case the `scipy` functions perform well. 

The function we will use is [solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html). Read the documentation first!

For more details on the nonlinear pendulum see [here](https://www.thphys.uni-heidelberg.de/~gasenzer/index.php?n1=teaching&n2=chaos). 

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.rcParams['legend.fancybox'] = True
import matplotlib.pyplot as plt
import scipy.integrate
import concurrent
import time
from pprint import pprint

## Set initial conditions and initialize trajectory

## Implement a "Pendulum" class.

This class implements the derivative of the state vector. This is another example of a "functor": a class that implements a `()` method, so you can use it like a function, but also use class variables to store a "state" of the function. We implement everything in a vectorized way, which will drastically speed up our computation in `solve_ivp` below. This will also accept all of the parameters in the dictionary that is passed to it and use `setattr` to create data members. 

In [ ]:
class Pendulum:
    def __init__ (self, 
                 gamma: float, 
                 omega_0: float, 
                 F_D: float, 
                 omega_D: float):
        self.gamma = gamma
        self.omega_0 = omega_0
        self.F_D = F_D
        self.omega_D = omega_D

    def __call__(self, t, state):
        flow = np.zeros_like(state)
        flow[0] = state[1]
        flow[1] = -self.omega_0**2 * np.sin(state[0]) - self.gamma * state[1] + self.F_D * np.cos(self.omega_D * t)    
        return flow

## Simulate and plot the trajectories

If you haven't done so yet, read the [solve_ivp documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html). Make sure to understand the optional arguments used below, including `vectorized`, `atol`, `rtol`, `method`, and `first_step`. 

We also set `dense_output=True`. Since RK45 is an adaptive algorithm, the resulting points are (probably) not equally spaced in time. Unfortunately, this causes problems for calculating the Poincaré section, which requires sampling the trajectory at exact multiples of the driving period. `dense_output` solves this by returning an *interpolation function*, allowing us to calculate the trajectory at whatever times we desire by interpolating the ODE solution's points. 

In [ ]:
# Chaotic pendulum
tmax = 2000.
theta_init = 0.0
omega_init = 0.0
method = "RK45"
pendul = {}
pendul["chaos"] = Pendulum(
    gamma = 0.5,
    omega_0 = 1.0, # natural frequency, not initial value!
    F_D = 1.15,
    omega_D = 2/3
)
res = scipy.integrate.solve_ivp(fun = pendul["chaos"],
                                t_span = (0., tmax),
                                y0 = [theta_init, omega_init], 
                                vectorized = True, 
                                atol = 1e-8,
                                rtol = 1e-6,
                                method = method,
                                first_step = 0.001, 
                                dense_output=True)
# Results storage
t = {}
states = {}
theta = {}
omega = {}
sol = {}
t["chaos"] = res.t
states["chaos"] = res.y
theta["chaos"] = states["chaos"][0]
omega["chaos"] = states["chaos"][1]
sol["chaos"] = res.sol

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 6))
ax[0].plot(t["chaos"], theta["chaos"])
ax[0].set_ylabel(r"$\theta(t)$ (radians)")
ax[0].text(1750, 25, "Chaos", size="large")

ax[1].plot(t["chaos"], omega["chaos"])
ax[1].set_xlabel("t (seconds)")
ax[1].set_ylabel(r"$\omega(t)$ (radians)")


In [ ]:
# Linear pendulum
tmax = 2000.
theta_init = 0.0
omega_init = 0.0
method = "RK45"
pendul["linear"] = Pendulum(
    gamma = 0.5, 
    omega_0 = 1.0, 
    F_D = 0.6, 
    omega_D = 2./3
)

res = scipy.integrate.solve_ivp(fun = pendul["linear"],
                                t_span = (0., tmax),
                                y0 = [theta_init, omega_init], 
                                vectorized = True, 
                                atol = 1e-10,
                                rtol = 1e-8,
                                method = method,
                                first_step = 0.001, 
                                dense_output=True)
t["linear"] = res.t
states["linear"] = res.y
theta["linear"] = states["linear"][0]
omega["linear"] = states["linear"][1]
sol["linear"] = res.sol

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 6))
ax[0].plot(t["linear"], theta["linear"])
ax[0].set_ylabel(r"$\theta(t)$ (radians)")
ax[0].text(175, 0.12, "Linear", size="large")

ax[1].plot(t["linear"], omega["linear"])
ax[1].set_xlabel("t (seconds)")
ax[1].set_ylabel(r"$\omega(t)$ (radians)")


In [ ]:
# 2-cycle pendulum
tmax = 1000.
theta_init = 0.0
omega_init = 0.0
method = "RK45"
pendul["twocycle"] = Pendulum(
    gamma = 0.5, 
    omega_0 = 1.0, 
    F_D = 1.075, 
    omega_D = 2./3
)

res = scipy.integrate.solve_ivp(fun = pendul["twocycle"],
                                t_span = (0., tmax),
                                y0 = [theta_init, omega_init], 
                                vectorized = True, 
                                atol = 1e-8,
                                rtol = 1e-6,
                                method = method,
                                first_step = 0.001, 
                                dense_output=True)
t["twocycle"] = res.t
states["twocycle"] = res.y
theta["twocycle"] = states["twocycle"][0]
omega["twocycle"] = states["twocycle"][1]
sol["twocycle"] = res.sol

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 6))
ax[0].plot(t["twocycle"], theta["twocycle"])
ax[0].set_ylabel(r"$\theta(t)$ (radians)")
ax[0].text(175, 24, "Two cycle", size="large")

ax[1].plot(t["twocycle"], omega["twocycle"])
ax[1].set_xlabel("t (seconds)")
ax[1].set_ylabel(r"$\omega(t)$ (radians)")


## Define a Poincare map

Now we will define a Poincare map defined by the steady-state period. If there is no driving force, this will be the period corresponding to the natural frequency $\omega_0$. If there is a driving force, it is the period corresponding to the driving frequency $\omega_D$. 

The Poincare section will be the state vector (in this case, $(\theta, \omega)$), evaluated after each period has elapsed. We already have the entire time series (plotted above), so we just need to do some array programming to get the values we need. 

We will first "wrap" the angle $\theta$ from $\left(-\pi, \pi\right]$. We will also ignore the transients by removing times below a threshold (in this case, we require $t > 100$ s). The Poincare map is therefore constructed by the state vector at each period, as described below. 


In [ ]:
theta1 = {}
for case in ["linear", "twocycle", "chaos"]:
    if pendul[case].omega_D > 0.0:
        T_p = 2*np.pi / pendul[case].omega_D
    else:
        T_p = 2*np.pi / pendul[case].omega_0
    
    
    theta1[case] = (theta[case] + np.pi) % (2 * np.pi ) - np.pi

### Plot wrapped angle time series

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 12))
for i, case in enumerate(["linear", "twocycle", "chaos"]):
    ax[i].plot(t[case], theta1[case])
    ax[i].set_xlabel("t (seconds)")
    ax[i].set_ylabel(r"$\theta(t)$ (radians)")


### Plot trajectory (ignoring transients)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
transient_mask = {}
for i, case in enumerate(["linear", "twocycle", "chaos"]):
    transient_mask[case] = t[case][1:] > 300.
    ax[i].plot(theta1[case][1:][transient_mask[case]], omega[case][1:][transient_mask[case]])
    ax[i].set_title("Total trajectory")
    ax[i].set_xlabel(r"wrapped angle (radians)")
    ax[i].set_ylabel(r"angular velocity (radians/s)")


### Plot the Poincare section itself 

To get the number of periods elapsed ("period number" $t/T_p$), we divide the time by the period with "floor division". This also conveniently works for floating point numbers in `python`! We can then look at $t_i/T_p$ and compare to $t_{i-1}/T_p$. If the previous period number is different from this period number, we are at the desired point for the Poincare map. 

In [ ]:
poincaremap = {}
ptheta = {}
pomega = {}

t_transient = 300.
for case in ["linear", "twocycle", "chaos"]:
    if pendul[case].omega_D > 0.0:
        T_p = 2 * np.pi / pendul[case].omega_D
    else:
        T_p = 2 * np.pi / pendul[case].omega_0
    n = np.arange(np.ceil(t_transient / T_p), np.floor(t[case][-1] / T_p) + 1)
    th_s, om_s = sol[case](n * T_p)                      # interpolate exactly at t = n T_p
    ptheta[case] = (th_s + np.pi) % (2 * np.pi) - np.pi
    pomega[case] = om_s

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for i, case in enumerate(["linear", "twocycle", "chaos"]):
    ax[i].scatter(ptheta[case], pomega[case])
    ax[i].set_xlim(-np.pi, np.pi)
    ax[i].set_ylim(-np.pi, np.pi)
    ax[i].set_xlabel(r"wrapped angle (radians)")
    ax[i].set_ylabel(r"angular velocity (radians/s)")

You might also want a programmatic way to determine how many cycles are present in your Poincaré section. It turns out our simulation is good enough that the trajectory returns to the same point each period, with a fair number of decimal points. So, we can use something like:

In [ ]:
def count_cycles(ptheta, pomega, tol=1e-3, p_max=32):
    """Smallest p such that the section repeats every p periods; 0 if none (chaos?)."""
    for p in range(1, p_max + 1):
        dth = np.angle(np.exp(1j * (ptheta[p:] - ptheta[:-p])))   # angle difference, wrapped
        if np.all(np.abs(dth) < tol) and np.all(np.abs(pomega[p:] - pomega[:-p]) < tol):
            return p
    return 0
    
for case in ["linear", "twocycle", "chaos"]:
    print(f"{case} : count_cycles = {count_cycles(ptheta[case], pomega[case])}")

## Toward bifurcation diagram

You need to be able to plot the Poincare section above as a function of a parameter. So, you need to "smoosh" across one axis. To do this, you really just need the 1-d vector `ptheta`, but then make a scatter plot all with the same $x$ axis value, $F_D$. 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
bif_xvals = np.full_like(ptheta["chaos"], pendul["chaos"].F_D)
ax.scatter(bif_xvals, ptheta["chaos"], color='k', marker='.', s=1)
ax.set_xlim(1.0, 1.2)
ax.set_ylim(-np.pi, np.pi)
ax.set_xlabel(r"$F_D$", fontsize="large")
ax.set_ylabel(r"$\theta_P$", fontsize="large")

Obviously, you need to vary the value and compute the appropriate Poincare map. 

## Bifurcation diagram

Your turn! You have all the code you need to generate the bifurcation diagram. Using the code from above, generate the $\theta_P$ values for an array of $F_D$ values, and plot. 

You can optionally try a couple variations:

1. Use multiprocessing to speed up the calculation
2. Remember, the Poincaré section is fairly sensitive to the initial conditions. This can result in a "fork" in the bifurcation plot that actually corresponds to mirror image trajectories, instead of true cycle doubling. A couple ideas to suppress this unwanted fact and focus just on $F_D$ dependence:
    1. Color the points by the number of cycles detected by the `count_cycle()` function
    2. Start each new calculation from the previous calculation's final state. Note: this is incompatible with multiprocessing. 


In [ ]:
FDvals = np.linspace(1.0, 1.3, 60)

xvals = []
yvals = []
for FD in FDvals:
    pendulum = Pendulum(
                    gamma=0.5, 
                    omega_0=1.0, 
                    F_D=FD, 
                    omega_D = 2./3
                )
    # Call solve_ivp()

    # Convert trajectory to Poincaré section

    # Project Poincaré section from (theta, omega) to (omega) and store output

# Plot output


In [ ]:
'''
Note: this is a good use case for multiprocessing. Here is an example.
MacOS users: add the following to the arguments of ProcessPoolExecutor():
with concurrent.futures.ProcessPoolExecutor(mp_context=mp.get_context("fork"), max_workers=nworkers) as executor:
'''

import multiprocessing as mp

def some_func(x):
    # Dummy function
    time.sleep(1)
    return x, np.ones(10) * x

print("Test one call")
t1 = time.time()
some_func(0.)
t2 = time.time()
print(f"One call, T = {t2-t1} s")

print(f"cpu_count = {os.cpu_count()}")
nworkers = os.cpu_count() # or os.cpu_count()
with concurrent.futures.ProcessPoolExecutor(max_workers=nworkers) as executor:
    xlist = np.arange(0, 3*os.cpu_count(), 1)
    results = {}
    t1 = time.time()
    for x, result in executor.map(some_func, xlist):
        results[x] = result
    t2 = time.time()
    print(f"{len(xlist)} calls with {nworkers} workers: T = {t2-t1} s")
pprint(results)

In [ ]:
# Define a few parameters to check some typical trajectories. 
'''
params_linear = {
    'method':'RK45',
    'tmax':200.,
    'theta_init':0.1,
    'omega_init':0.0,
    'gamma':0.0,
    'omega0':1.0,
    'F_D':0,
    'omegaD':0
}
params_twocycle ={
    'method':'RK45',
    'tmax':200.,
    'theta_init':0.0,
    'omega_init':0.0,
    'gamma':0.5,
    'omega0':1.0,
    'F_D':1.07,
    'omegaD':2/3
}
params_chaos ={
    'method':'RK45',
    'tmax':2000.,
    'theta_init':0.0,
    'omega_init':0.0,
    'gamma':0.5,
    'omega0':1.0,
    'F_D':1.15,
    'omegaD':2/3
}
'''